# Experiment 2 — density clustering and rejection accounting

Version 2. Parameters derive from distances, without inspecting true labels. Purity is always accompanied by coverage.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jdr.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd


In [ ]:
from src.jdr import JDRConfig, build_distance_matrix
from src.evaluation import load_metadata, aligned_labels, evaluate_clustering
from src.JDRKMedoids import JDRKMedoids
from src.JDRDBSCAN import JDRDBSCAN
files = sorted((ROOT / 'data').glob('*RRLYR*.dat'))[:12]
# Low-frequency numerical demonstration, NOT an RR Lyrae period-search band.
config = JDRConfig(f_min=0.001, f_max=0.03, n_frequencies=4097)
D = build_distance_matrix(files, config)
metadata = load_metadata(sorted((ROOT / 'notebooks').glob('ident*.dat')))
y_true = aligned_labels(files, metadata)
assert np.allclose(D, D.T) and np.isfinite(D).all()


In [ ]:
eps = float(np.quantile(np.sort(D, axis=1)[:, 3], 0.7))
model = JDRDBSCAN(eps=eps, min_samples=4).fit(D)
evaluate_clustering(y_true, model.labels_)
